# 2.Office文档加载器

知识点讲解：Office 文档加载器的使用

1. Office 文档加载器家族
   - UnstructuredExcelLoader       : 加载 Excel（.xlsx, .xls）
   - UnstructuredWordDocumentLoader: 加载 Word（.docx, .doc）
   - UnstructuredPowerPointLoader  : 加载 PowerPoint（.pptx, .ppt）
   - 三者都基于 unstructured 库，能解析复杂的 Office 格式并提取文本

2. mode 参数与「元素」分离
   - "single"（默认）：将整个文件作为一个 Document
   - "elements"：按文档元素分割（单元格、段落、幻灯片），适合细粒度检索
   - Office 类非结构化加载器使用极简：一般只传对应文档路径即可
   - 需要区分文档中的元素时传 mode="elements"，
     但文档明确建议一般不使用——原因同 Markdown：加载阶段分割无法保证块大小一致性

3. 各加载器特点
   - Excel      : 支持多工作表，elements 模式下每个单元格可作为一个元素，适合结构化表格数据
   - Word       : 保留段落结构，可提取标题、正文、列表，自动去除格式只留文本
   - PowerPoint : 按幻灯片或元素提取，含标题、文本框、备注，metadata 含幻灯片编号

4. 依赖库与精确安装命令
   - 需要 unstructured 库及其 Office 相关依赖：
     * Excel : pip install unstructured openpyxl pandas
     * PPT   : pip install unstructured python-magic python-pptx
     * Word  : pip install unstructured python-docx
   - 注意 PPT 额外需要 python-magic（做文件类型嗅探），Excel 额外需要 pandas

5. 典型输出示例（PPT + mode="elements"）
   ppt_loader = UnstructuredPowerPointLoader("./章节介绍.pptx", mode="elements")
   documents = ppt_loader.load()
   [Document(page_content='LangChain RAG应用开发组件深入学习',
             metadata={'source': './章节介绍.pptx', 'category_depth': 1,
                       'file_directory': '.', 'filename': '章节介绍.pptx',
                       'last_modified': '2024-07-20T11:44:28', 'page_number': 1,
                       'languages': ['zho', 'kor'],
                       'filetype': 'application/vnd.openxmlformats-officedocument.presentationml.presentation',
                       'category': 'Title'}),
    Document(page_content='章节介绍', metadata={..., 'category': 'Title'})]
   17
   LangChain RAG应用开发组件深入学习
   - metadata 补充字段：
     * category_depth : 元素在文档结构中的层级深度（一级标题为 1）
     * page_number    : PPT 中即幻灯片页码
     * languages      : 识别出的语言列表，'zho' 中文、'kor' 韩文
     * filetype       : Office 文件的完整 MIME 类型（OOXML 规范）

6. 应用延伸：ChatPDF
   - 利用 unstructured 的办公文档加载能力，配合 LLM 即可实现 2023 年爆火的 ChatPDF 功能：
     上传特定 PDF，让 LLM 实现对该 PDF 的问答
   - 本质链路：Office/PDF 加载器 → 分割器 → 向量库 → 检索器 → LLM


In [ ]:
from langchain_community.document_loaders import (
    UnstructuredPowerPointLoader,
)

In [ ]:
ppt_loader = UnstructuredPowerPointLoader("./章节介绍.pptx")
documents = ppt_loader.load()
print(documents)
print(len(documents))
print(documents[0].metadata)